In [30]:
#Install required dependencies (Wolfram Engine also needs to be installed and activated on the machine too)
#!pip install wolframclient
#!pip install Jinja2 #required for latex tables

#Import relevant libraries
import pandas as pd
import numpy as np
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wlexpr
import atexit 

# Reading OpenFMS outputs
This part of the code reads a .txt file containing the parameters of overlapping static TBFs in regions of constant and changing SOC for various model system variants. The file also contains the outputted values of the SOC term betwen TBFs using both the SPA0 and SPA1 within GAIMS, calculated by OpenFMS

In [31]:
with open("SPA1_verify.txt", "r") as f:
    text = f.readlines()

def extract_floats(var_name, lines):
    results = []
    for line in lines:
        parts = line.split()
        if parts and parts[0] == var_name:
            results.append(float(parts[1]))
    return results

def extract_complex(var_name, lines):
    results = []
    for line in lines:
        parts = line.split()
        if parts and parts[0] == var_name:
            inner_text = line.split('(')[1].split(')')[0] #Finds complex within brackets
            real_part, imag_part = inner_text.split(',')
            results.append(complex(float(real_part), float(imag_part)))
    return results

def extract_rsigs(lines):
    results = []
    for i, line in enumerate(lines):
        if "########DDDD#############" in line: #Rsig values are above this line
            value = lines[i-1].strip()
            results.append(float(value))
    return results

#Put inputs and outputs into separate dictionarys
inputs = {
    "rsigs": extract_rsigs(text),
    "xi_vals": extract_floats('x_i', text),
    "xj_vals": extract_floats('x_j', text),
    "pi_vals": extract_floats('p_i', text),
    "pj_vals": extract_floats('p_j', text),
    "phasei_vals": extract_floats('phase_i', text),
    "phasej_vals": extract_floats('phase_j', text)
}

openfms_outputs = {
    "spa0": extract_complex('SPA0', text),
    "spa1_code": extract_complex('SPA1', text),
    "overlap_code": extract_complex('S_j', text),
    "dSOC_code": extract_complex('dS0C', text),
    "rho_code": extract_complex('roe', text),
}

In [32]:
#Make a list of centroid positions between TBFs
inputs["xc_vals"] = [0] * len(inputs["xi_vals"])
for i in range(0,len(inputs["xi_vals"])):
    inputs["xc_vals"][i] = (inputs["xi_vals"][i]+inputs["xj_vals"][i])/2

# Wolfram Engine Calculations
This part uses the parameters of the overlapping TBFs and calculates the first order term of the SPA1 for the SOC term between the TBFs as well as the exact coupling value.

In [ ]:
#Opens Wolfram Enginge Session and quits once all calculations are done
session = WolframLanguageSession()
atexit.register(session.terminate)

def parse_w(val): #Formats Wolfram outputs based on if they are complex or real numbers
    if hasattr(val, 'args') and len(val.args) == 2:
        c = complex(float(val.args[0]), float(val.args[1]))
    else:
        c = complex(val)
    return c.real if c.imag == 0.0 else c

wolfram_outputs = {
    "overlap": [], 
    "phase": [], 
    "spa1_integral": [], 
    "rho": [],
    "dSOC": [], 
    "exact": []
}

#Loops over the pairs of overlapping TBFs
for i in range(len(inputs["xi_vals"])):
    
    #Uses input values for a given case (different Rsig and changing/constant region)
    rsig = inputs["rsigs"][i]
    xi = inputs["xi_vals"][i]
    xj = inputs["xj_vals"][i]
    pi = inputs["pi_vals"][i]
    pj = inputs["pj_vals"][i]
    phasei = inputs["phasei_vals"][i]
    phasej = inputs["phasej_vals"][i]
    xc = inputs["xc_vals"][i]

    #Bounds for integration, as Gaussian widths are 20, ensures integration over whole range of where gaussians overlap
    lower_bound = min(xi, xj) - 10
    upper_bound = max(xi, xj) + 10

    #Equations
    eqs = {
        "overlap": f"Quiet[NIntegrate[Exp[-20*(x-{xi})^2 - I*{pi}*(x-{xi}) - 20*(x-{xj})^2 + I*{pj}*(x-{xj})], {{x, {lower_bound}, {upper_bound}}}, WorkingPrecision -> 50, PrecisionGoal -> 30, AccuracyGoal -> 30, Method -> {{\"GlobalAdaptive\", \"SymbolicProcessing\" -> 0}}]]",
        "phase": f"N[Sqrt[40/Pi] * Exp[-I*{phasei} + I*{phasej}], 50]",
        "spa1_integral": f"Quiet[NIntegrate[Exp[-20*(x-{xi})^2 - 20*(x-{xj})^2 - I*{pi}*(x-{xi}) + I*{pj}*(x-{xj})] * (x-{xc}), {{x, {lower_bound}, {upper_bound}}}, WorkingPrecision -> 50, PrecisionGoal -> 30, AccuracyGoal -> 30, Method -> {{\"GlobalAdaptive\", \"SymbolicProcessing\" -> 0}}]]",
        "rho": f"N[-I*({pi} - {pj})/80, 50]"
    }
    
    #Breaks up integration into the arguments of the sigmoid function
    clamped_sigmoid = f"Piecewise[{{{{1, x <= {rsig}-1}}, {{-1, x >= {rsig}+1}}}}, 4*((x-{rsig})/2)^3 - 3*((x-{rsig})/2)]"
    eqs["exact"] = f"Quiet[NIntegrate[Exp[-20*(x-{xi})^2 - 20*(x-{xj})^2 - I*{pi}*(x-{xi}) + I*{pj}*(x-{xj})] * ({clamped_sigmoid}), {{x, {lower_bound}, {upper_bound}}}, WorkingPrecision -> 50, PrecisionGoal -> 30, AccuracyGoal -> 30, Method -> {{\"GlobalAdaptive\", \"SymbolicProcessing\" -> 0}}]]"
    
    if i % 2 == 0: #Changing region
        eqs["dSOC"] = f"N[ReplaceAll[D[4*((x - {rsig})/2)^3 - 3*((x - {rsig})/2), x], x -> {xc}], 50]"
    else: #Constant region
        eqs["dSOC"] = f"N[ReplaceAll[D[-1, x], x -> {xc}], 50]"
        
    #Parses Equations to the wolfram alpha with a given set of inputs and appends to dictionary
    for name, eq_str in eqs.items():
        raw_result = session.evaluate(wlexpr(eq_str)) #Calculation
        wolfram_outputs[name].append(parse_w(raw_result))

From the Wolfram Engine outputs, format into the terms needed to compare to OpenFMS

In [ ]:
#Formats lists into numpy array with scientific notation
def conv_array(lst):
    sn_list = []
    for z in lst:
        if isinstance(z, str):
            z = z.replace("×10^", "e")
        sn_list.append(complex(z))
    return np.array(sn_list, dtype=complex)

for i in wolfram_outputs:
    wolfram_outputs[i] = conv_array(wolfram_outputs[i])

for i in openfms_outputs:
    openfms_outputs[i] = conv_array(openfms_outputs[i])

z = 0.0005 + 0.0005j

#Calculates relevant terms from Wolfram outputs
overlap = wolfram_outputs["overlap"] * wolfram_outputs["phase"]
exact = z * wolfram_outputs["exact"] * wolfram_outputs["phase"]
dSOC = z * wolfram_outputs["dSOC"]

SPA1_analytical = overlap *  wolfram_outputs["rho"]
SPA1_numerical = wolfram_outputs["spa1_integral"] * wolfram_outputs["phase"]
spa1 = (dSOC * SPA1_numerical) + openfms_outputs["spa0"]

#Calculates relevant terms from Openfms outputs to compare to Wolfram outputs
first_integral_code = openfms_outputs["overlap_code"] * openfms_outputs["rho_code"]
spa1_code = openfms_outputs["spa1_code"] + openfms_outputs["spa0"]

# Tables
Tables comparing OpenFMS and Wolfram Engine values (in Appendix)

In [35]:
pd.set_option('display.float_format', '{:.5E}'.format)
region = ['Changing','Constant','Changing','Constant','Changing','Constant']

In [36]:
table_list = [inputs["rsigs"],region,openfms_outputs["overlap_code"].imag,overlap.imag,openfms_outputs["overlap_code"].real,overlap.real]
df = pd.DataFrame(table_list,index=['Sign Change / bohr','Region','Imag(Overlap) - OpenFMS','Imag(Overlap) - Wolfram','Real(Overlap) - OpenFMS','Real(Overlap) - Wolfram']).T
df.sort_values(by=['Region'])

,Sign Change / bohr,Region,Imag(Overlap) - OpenFMS,Imag(Overlap) - Wolfram,Real(Overlap) - OpenFMS,Real(Overlap) - Wolfram
0,8.00000E+00,Changing,-6.28469E-04,-6.28469E-04,-8.20914E-04,-8.20914E-04
2,9.50000E+00,Changing,-6.68636E-01,-6.68636E-01,6.05264E-01,6.05264E-01
4,1.00000E+01,Changing,-3.10386E-02,-3.10386E-02,9.96447E-01,9.96447E-01
1,8.00000E+00,Constant,-1.96151E-03,-1.96151E-03,-1.60078E-02,-1.60078E-02
3,9.50000E+00,Constant,-1.96151E-03,-1.96151E-03,-1.60078E-02,-1.60078E-02
5,1.00000E+01,Constant,-1.96808E-03,-1.96808E-03,-1.60104E-02,-1.60104E-02


In [37]:
table_list = [inputs["rsigs"],region,openfms_outputs["rho_code"].imag,wolfram_outputs["rho"].imag]
df = pd.DataFrame(table_list,index=['Sign Change / bohr','Region','Imag(Rho) - OpenFMS','Imag(Rho) - Wolfram']).T
df.sort_values(by=['Region'])

,Sign Change / bohr,Region,Imag(Rho) - OpenFMS,Imag(Rho) - Wolfram
0,8.00000E+00,Changing,2.31559E-01,2.31559E-01
2,9.50000E+00,Changing,4.78033E-02,4.78033E-02
4,1.00000E+01,Changing,-8.74481E-03,-8.74481E-03
1,8.00000E+00,Constant,-1.53521E-01,-1.53521E-01
3,9.50000E+00,Constant,-1.53521E-01,-1.53521E-01
5,1.00000E+01,Constant,-1.53520E-01,-1.53520E-01


In [38]:
table_list = [inputs["rsigs"],region,first_integral_code.real,SPA1_analytical.real,SPA1_numerical.real]
df = pd.DataFrame(table_list,index=['Sign Change / bohr','Region','Real(1st order integral) - code','Real(1st order integral) - Wolfram Vanicek','Real(1st order integral) - Wolfram Numerical']).T
df.sort_values(by=['Region'])

,Sign Change / bohr,Region,Real(1st order integral) - code,Real(1st order integral) - Wolfram Vanicek,Real(1st order integral) - Wolfram Numerical
0,8.00000E+00,Changing,1.45527E-04,1.45527E-04,1.45527E-04
2,9.50000E+00,Changing,3.19630E-02,3.19630E-02,3.19630E-02
4,1.00000E+01,Changing,-2.71427E-04,-2.71427E-04,-2.71427E-04
1,8.00000E+00,Constant,-3.01133E-04,-3.01133E-04,-3.01133E-04
3,9.50000E+00,Constant,-3.01133E-04,-3.01133E-04,-3.01133E-04
5,1.00000E+01,Constant,-3.02140E-04,-3.02140E-04,-3.02140E-04


In [39]:
table_list = [inputs["rsigs"],region,first_integral_code.imag,SPA1_analytical.imag,SPA1_numerical.imag]
df = pd.DataFrame(table_list,index=['Sign Change / bohr','Region','Imag(1st order integral) - OpenFMS','Imag(1st order integral) - Wolfram Vanicek','Imag(1st order integral) - Wolfram Numerical']).T
df.sort_values(by=['Region'])

,Sign Change / bohr,Region,Imag(1st order integral) - OpenFMS,Imag(1st order integral) - Wolfram Vanicek,Imag(1st order integral) - Wolfram Numerical
0,8.00000E+00,Changing,-1.90090E-04,-1.90090E-04,-1.90090E-04
2,9.50000E+00,Changing,2.89336E-02,2.89336E-02,2.89336E-02
4,1.00000E+01,Changing,-8.71374E-03,-8.71374E-03,-8.71374E-03
1,8.00000E+00,Constant,2.45754E-03,2.45754E-03,2.45754E-03
3,9.50000E+00,Constant,2.45754E-03,2.45754E-03,2.45754E-03
5,1.00000E+01,Constant,2.45792E-03,2.45792E-03,2.45792E-03


In [40]:
table_list = [inputs["rsigs"],region,openfms_outputs["dSOC_code"].imag,dSOC.imag,openfms_outputs["dSOC_code"].real,dSOC.real]
df = pd.DataFrame(table_list,index=['Sign Change / bohr','Region','Imag(dSOC) - Code','Imag(dSOC)- Wolfram','Real(dSOC) - Code','Real(dSOC) - Wolfram',]).T
df.sort_values(by=['Region'])

,Sign Change / bohr,Region,Imag(dSOC) - Code,Imag(dSOC)- Wolfram,Real(dSOC) - Code,Real(dSOC) - Wolfram
0,8.00000E+00,Changing,-6.12590E-04,-6.12590E-04,-6.12590E-04,-6.12590E-04
2,9.50000E+00,Changing,-7.47115E-04,-7.47115E-04,-7.47115E-04,-7.47115E-04
4,1.00000E+01,Changing,-7.43955E-04,-7.43955E-04,-7.43955E-04,-7.43955E-04
1,8.00000E+00,Constant,0.00000E+00,0.00000E+00,0.00000E+00,0.00000E+00
3,9.50000E+00,Constant,0.00000E+00,0.00000E+00,0.00000E+00,0.00000E+00
5,1.00000E+01,Constant,0.00000E+00,0.00000E+00,0.00000E+00,0.00000E+00


Tables Comparing SPA0, SPA1 and Exact values for SOC matrix elements for Results

In [41]:
df_real = pd.DataFrame({
    'x_c / bohr': inputs["rsigs"],
    'Region': region,
    'Component': 'Real',
    'SPA0 / Ha': openfms_outputs["spa0"].real,
    'SPA1 / Ha': spa1_code.real,
    'Exact / Ha': exact.real
})

df_imag = pd.DataFrame({
    'x_c / bohr': inputs["rsigs"],
    'Region': region,
    'Component': 'Imaginary',
    'SPA0 / Ha': openfms_outputs["spa0"].imag,
    'SPA1 / Ha': spa1_code.imag,
    'Exact / Ha': exact.imag
})

df = pd.concat([df_real, df_imag], ignore_index=True)

def format_table(region_name):
    return(df[df['Region'] == region_name].drop(columns=['Region'])
        .sort_values(by=['Component', 'x_c / bohr'], ascending=[False, True]))

df_chang = format_table('Changing')
df_const = format_table('Constant')

In [42]:
df_const
#print(df_const.to_latex())

,x_c / bohr,Component,SPA0 / Ha,SPA1 / Ha,Exact / Ha
1,8.00000E+00,Real,7.02315E-06,7.02315E-06,7.02315E-06
3,9.50000E+00,Real,7.02315E-06,7.02315E-06,7.02315E-06
5,1.00000E+01,Real,7.02115E-06,7.02115E-06,7.02115E-06
7,8.00000E+00,Imaginary,8.98466E-06,8.98466E-06,8.98466E-06
9,9.50000E+00,Imaginary,8.98466E-06,8.98466E-06,8.98466E-06
11,1.00000E+01,Imaginary,8.98922E-06,8.98922E-06,8.98922E-06


In [43]:
df_chang
#print(df_chang.to_latex())

,x_c / bohr,Component,SPA0 / Ha,SPA1 / Ha,Exact / Ha
0,8.00000E+00,Real,5.80067E-08,-1.47589E-07,-1.46401E-07
2,9.50000E+00,Real,-5.91770E-05,-6.14403E-05,-6.08083E-05
4,1.00000E+01,Real,-6.89958E-05,-7.52765E-05,-7.43380E-05
6,8.00000E+00,Imaginary,4.36873E-07,4.64171E-07,4.83483E-07
8,9.50000E+00,Imaginary,2.94383E-06,-4.25529E-05,-4.20469E-05
10,1.00000E+01,Imaginary,-6.48273E-05,-5.81427E-05,-5.74192E-05
